___
## Modify a table with sensitive data 

Step 1: Create a backup table

In [5]:
import sqlite3
from datetime import datetime

def create_table_backup(table_name, backup_sufix=None):
    """ 
    This function creates timestamped backup of any table for safety

    parameters: 
    table_name(str): Name of the table to backup
    backup_sufix(str): Custom suffix for backup table name 
    """
    conn = None
    # The main operation that might fail
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()

        # Generate the backup table with timestamp
        if backup_sufix:
            backup_table = f"{table_name}_BACKUP_{backup_sufix}"
        else:
            timestamp = datetime.now().strftime("%m_%d_%y")
            backup_table = f"{table_name}_BACKUP_{timestamp}"
        
        # Execute backup
        cursor.execute(f"""
        CREATE TABLE {backup_table} AS 
        SELECT * FROM {table_name}
        """)

        # Verify backup was created
        cursor.execute(f"SELECT COUNT(*) FROM {backup_table}")
        record_count = cursor.fetchone()[0]

        conn.commit()

        print(f"☑️ Backup successful: {backup_table}")
        print(f"📊 Records backed up: {record_count}")

        return backup_table

    # Handle database errors specifically
    except sqlite3.Error as e:  
        print (f"🚫 Backup failed for {table_name}: {e}")
        if conn:
            conn.rollback()
        return None
    
    # Ensure connection always closes even if error occurs
    finally:
        if conn:
            conn.close()


# --- PRACTICE USAGE ---
if __name__ == "__main__":
    # basic usage
    backup_name = create_table_backup("OBSERVACION_LOTE")


☑️ Backup successful: OBSERVACION_LOTE_BACKUP_10_31_25
📊 Records backed up: 176


Step 2: Modofy the date format

In [4]:
import sqlite3

def flip_date_format (table_name, date_column='FECHA_OBSERVACION'):
    """ 
    Flip date format from dd-mm-yy to mm-dd-yy for improved sorting

    Parameters:
    table_name: Name of the table to update
    date_column: Name of the date field
    """
    conn = None
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()

        # SQL to update the date format
        update_query = f"""
            UPDATE {table_name}
            SET {date_column} = 
                substr({date_column}, 4, 2) || '-' ||
                substr({date_column}, 1, 2) || '-' ||
                substr({date_column}, 7, 2)
            WHERE {date_column} LIKE '__-__-__'
        """

        cursor.execute(update_query)
        rows_updated = cursor.rowcount

        conn.commit()

        print(f"👍🏽 Date format flipped in {table_name}, {date_column}")
        print(f"📊 Rows updated: {rows_updated}")

        return rows_updated
    
    except sqlite3.Error as e: 
        print (f"Error flipping the dates: {e}")
        if conn:
            conn.rollback()
        return None
    
    finally:
        if conn:
            conn.close()

# --- PRACTICE ---
if __name__ == "__main__":
    flip_date_format('OBSERVACION_LOTE_BACKUP_10_31_25')

👍🏽 Date format flipped in OBSERVACION_LOTE_BACKUP_10_31_25, FECHA_OBSERVACION
📊 Rows updated: 176


step 3: update field values